In [ ]:
!pip -q install numpy pandas scikit-learn joblib ipywidgets

print("All required libraries are installed.")

All required libraries are installed.


In [ ]:
import pandas as pd
from functools import lru_cache

# Board representation:
# X = -1, empty = 0, O = 1
WIN_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]

def winner(board):
    for a, b, c in WIN_LINES:
        if board[a] != 0 and board[a] == board[b] == board[c]:
            return board[a]
    return 0

def terminal(board):
    return winner(board) != 0 or 0 not in board

@lru_cache(maxsize=None)
def minimax(board):
    result = winner(board)

    if result == 1:
        return 1
    if result == -1:
        return -1
    if 0 not in board:
        return 0

    # X starts. O plays when X has one extra mark.
    player = 1 if board.count(-1) > board.count(1) else -1
    scores = []

    for i, cell in enumerate(board):
        if cell == 0:
            child = list(board)
            child[i] = player
            scores.append(minimax(tuple(child)))

    return max(scores) if player == 1 else min(scores)

def best_move(board):
    candidates = []

    for i, cell in enumerate(board):
        if cell == 0:
            child = list(board)
            child[i] = 1  # Computer plays O
            candidates.append((minimax(tuple(child)), i))

    best_score = max(score for score, _ in candidates)

    # Deterministic choice when multiple moves are equally good.
    return next(
        i for score, i in candidates
        if score == best_score
    )

rows = []
seen = set()

def generate(board):
    if board in seen:
        return

    seen.add(board)

    if terminal(board):
        return

    x_count = board.count(-1)
    o_count = board.count(1)

    # Record positions where it is O's turn.
    if x_count == o_count + 1:
        rows.append(list(board) + [best_move(board)])

    player = -1 if x_count == o_count else 1

    for i, cell in enumerate(board):
        if cell == 0:
            child = list(board)
            child[i] = player
            generate(tuple(child))

generate((0,) * 9)

columns = [f"cell_{i}" for i in range(9)] + ["best_move"]
df = pd.DataFrame(rows, columns=columns)

df.to_csv("tic_tac_toe_dataset.csv", index=False)

print("Dataset generated successfully!")
print("Number of training examples:", len(df))
print("Dataset shape:", df.shape)
display(df.head(10))

Dataset generated successfully!
Number of training examples: 2097
Dataset shape: (2097, 10)


,cell_0,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,best_move
0,-1,0,0,0,0,0,0,0,0,4
1,-1,1,-1,0,0,0,0,0,0,4
2,-1,1,-1,1,-1,0,0,0,0,5
3,-1,1,-1,1,-1,1,0,-1,0,6
4,-1,1,-1,1,-1,-1,1,0,0,8
5,-1,1,-1,1,-1,0,1,-1,0,8
6,-1,1,-1,1,-1,-1,0,1,0,6
7,-1,1,-1,1,-1,-1,0,0,1,6
8,-1,1,-1,1,-1,0,0,-1,1,6
9,-1,1,-1,1,0,-1,0,0,0,8


In [ ]:
print("Dataset information:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nNumber of examples for each target move:")
print(df["best_move"].value_counts().sort_index())

print("\nExample board:")
example = df.iloc[0]

for row in range(3):
    values = [
        int(example[f"cell_{row * 3 + col}"])
        for col in range(3)
    ]
    print(values)

print("\nRecommended computer move:", int(example["best_move"]))

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2097 entries, 0 to 2096
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   cell_0     2097 non-null   int64
 1   cell_1     2097 non-null   int64
 2   cell_2     2097 non-null   int64
 3   cell_3     2097 non-null   int64
 4   cell_4     2097 non-null   int64
 5   cell_5     2097 non-null   int64
 6   cell_6     2097 non-null   int64
 7   cell_7     2097 non-null   int64
 8   cell_8     2097 non-null   int64
 9   best_move  2097 non-null   int64
dtypes: int64(10)
memory usage: 164.0 KB
None

Missing values:
cell_0       0
cell_1       0
cell_2       0
cell_3       0
cell_4       0
cell_5       0
cell_6       0
cell_7       0
cell_8       0
best_move    0
dtype: int64

Number of examples for each target move:
best_move
0    494
1    281
2    309
3    157
4    340
5    108
6    174
7     83
8    151
Name: count, dtype: int64

Example board:
[-1, 0, 0]
[0, 

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import random

# Human = -1, Computer = 1, Empty = 0
board = [0] * 9
game_over = False

WIN_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]

def winner(board):
    for a, b, c in WIN_LINES:
        if board[a] != 0 and board[a] == board[b] == board[c]:
            return board[a]
    return 0

def find_winning_move(player):
    for i in range(9):
        if board[i] == 0:
            board[i] = player
            result = winner(board)
            board[i] = 0

            if result == player:
                return i
    return None

status = widgets.HTML()

buttons = [
    widgets.Button(
        description=" ",
        layout=widgets.Layout(
            width="100px",
            height="100px",
            border="3px solid #1F2937"
        ),
        style=widgets.ButtonStyle(
            font_weight="bold",
            font_size="36px",
            text_color="#111827"
        )
    )
    for _ in range(9)
]

grid = widgets.GridBox(
    children=buttons,
    layout=widgets.Layout(
        grid_template_columns="repeat(3, 100px)",
        grid_gap="6px"
    )
)

restart_button = widgets.Button(
    description="RESTART GAME",
    button_style="success",
    layout=widgets.Layout(width="200px", height="45px"),
    style=widgets.ButtonStyle(font_weight="bold")
)

def computer_move():
    # 1. Win if possible
    move = find_winning_move(1)

    # 2. Block the human's immediate win
    if move is None:
        move = find_winning_move(-1)

    # 3. Take the center
    if move is None and board[4] == 0:
        move = 4

    # 4. Prefer a corner
    if move is None:
        corners = [i for i in [0, 2, 6, 8] if board[i] == 0]
        if corners:
            move = random.choice(corners)

    # 5. Choose any remaining empty cell
    if move is None:
        empty = [i for i in range(9) if board[i] == 0]
        if empty:
            move = random.choice(empty)

    if move is not None:
        board[move] = 1

def refresh_board():
    global game_over

    result = winner(board)

    if result == -1:
        game_over = True
        status.value = "<h2 style='color:#008000;font-weight:900'>🎉 YOU WIN!</h2>"
    elif result == 1:
        game_over = True
        status.value = "<h2 style='color:#D00000;font-weight:900'>🤖 COMPUTER WINS!</h2>"
    elif 0 not in board:
        game_over = True
        status.value = "<h2 style='color:#7209B7;font-weight:900'>IT'S A DRAW!</h2>"
    else:
        status.value = "<h2 style='color:#0057FF;font-weight:900'>YOUR TURN: X</h2>"

    symbols = {-1: "X", 0: " ", 1: "O"}

    for i, button in enumerate(buttons):
        button.description = symbols[board[i]]
        button.disabled = game_over or board[i] != 0

        if board[i] == -1:
            button.style.button_color = "#0066FF"
            button.style.text_color = "#FFFFFF"
        elif board[i] == 1:
            button.style.button_color = "#FF7A00"
            button.style.text_color = "#FFFFFF"
        else:
            button.style.button_color = "#FFFFFF"
            button.style.text_color = "#111827"

def handle_click(index):
    def callback(_):
        if game_over or board[index] != 0:
            return

        # Human plays
        board[index] = -1

        # Check whether the human has won or drawn
        if winner(board) != 0 or 0 not in board:
            refresh_board()
            return

        # Computer responds immediately
        computer_move()
        refresh_board()

    return callback

for i, button in enumerate(buttons):
    button.on_click(handle_click(i))

def restart_game(_):
    global board, game_over
    board = [0] * 9
    game_over = False
    refresh_board()

restart_button.on_click(restart_game)

refresh_board()
display(widgets.VBox([status, grid, restart_button]))